<a href="https://colab.research.google.com/github/nicolas-pueyo/ia-explicable/blob/main/notebooks/proyecto_iax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
  !pip install -q dtreeviz
!pip install interpret six graphviz pydotplus


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 48.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.1/800.1 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 28.1 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=4010811 sha256=f453b08ba907d90e8aa55ab0a1dcd2e2c111b50166774c57454f8ae8b7ff78cb
 

In [ ]:
import pandas as pd

base = "https://raw.githubusercontent.com/nicolas-pueyo/ia-explicable/main/data/"
train = pd.read_csv(base + "train.csv")
test = pd.read_csv(base + "test.csv")
dic = pd.read_excel(base + "data-dictionary.xlsx")

id_col = 'row_id'
target_col = 'payment_difficulty_next_month'

credit_y = train[target_col]
credit_X = train.drop(columns=[target_col, id_col])

categorical_cols = [
    'sex',
    'education',
    'marital_status',
    'repayment_assistance_plan'
]

numeric_cols = [col for col in credit_X.columns if col not in categorical_cols]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Definición del modelo lógico base (restringiendo profundidad para favorecer interpretabilidad)
tree_clf = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", tree_clf)
])

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ["accuracy", "roc_auc", "f1", "balanced_accuracy"]
cv_results = cross_validate(
    model_pipeline,
    credit_X,
    credit_y,
    cv=cv,
    scoring=scoring_metrics,
    return_train_score=False
)

for metric in scoring_metrics:
    mean_val = cv_results[f"test_{metric}"].mean()
    std_val = cv_results[f"test_{metric}"].std()
    print(f"{metric}: {mean_val:.4f} (+/- {std_val:.4f})")

In [ ]:
# Ajustar el pipeline a la totalidad del conjunto de entrenamiento
model_pipeline.fit(credit_X, credit_y)

# Obtener nombres de las características transformadas
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
fitted_tree = model_pipeline.named_steps["classifier"]

# Extracción textual de reglas lógicas
from sklearn.tree import export_text
tree_rules = export_text(fitted_tree, feature_names=list(feature_names))
print(tree_rules)

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# 1. Recuperar el clasificador ajustado y los nombres de las columnas transformadas
fitted_tree = model_pipeline.named_steps["classifier"]
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Limpiar prefijos automáticos ("cat__", "num__") para mejorar la legibilidad
clean_feature_names = [
    col.replace("cat__", "").replace("num__", "") for col in feature_names
]

# 2. Renderizar el gráfico del árbol
plt.figure(figsize=(22, 10), dpi=300)
plot_tree(
    fitted_tree,
    feature_names=clean_feature_names,
    class_names=["No Difficulty (0)", "Payment Difficulty (1)"],
    filled=True,
    rounded=True,
    fontsize=9,
    impurity=True,
    proportion=True,
)

plt.title("Logical Model: Decision Tree (Global Interpretation)", fontsize=14)
plt.tight_layout()

# 3. Guardar la figura en alta resolución para incluirla en el PDF del informe
plt.savefig("decision_tree.png", dpi=300, bbox_inches="tight")
plt.show()